In [4]:
!pip install pysmb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.2/85.2 KB 934.7 kB/s eta 0:00:0031m1.2 MB/s eta 0:00:01


In [12]:
from smb.SMBConnection import SMBConnection

# Kết nối tới Samba server
conn = SMBConnection(
    username="zaibachkhoa",
    password="s@mg11iongZon",
    my_name="laptop",      # tên máy hiện tại (tuỳ chọn)
    remote_name="server_pc",  # tên máy chủ Samba (có thể dùng IP)
    use_ntlm_v2=True
)
conn.connect("192.168.11.246", 445)  # 139)  # hoặc cổng 445

True

In [9]:
# # Tải file từ Samba share
# with open("local_copy.txt", "wb") as f:
#     conn.retrieveFile("GenAI", "/example.txt", f)

# Hoặc upload file lên server
with open("/media/zaibachkhoa/New Volume/__h2jsc_NESTECH/NSynth/nsynth-train/examples.json", "rb") as f:
    conn.storeFile("GenAI", "nsynth-train/examples.json", f)

print("✅ Sao chép file thành công!")


✅ Sao chép file thành công!


In [13]:
source = '/media/zaibachkhoa/New Volume/__h2jsc_NESTECH/NSynth/'
des = 'nsynth-train/audio/'
share_name = "GenAI"
import os


# --- Hàm kiểm tra file tồn tại ---
def file_exists(remote_path):
    try:
        dir_name = os.path.dirname(remote_path)
        file_name = os.path.basename(remote_path)
        files = conn.listPath(share_name, dir_name or "/")
        return any(f.filename == file_name for f in files)
    except Exception:
        return False

# --- Duyệt qua local folder ---
for root, dirs, files in os.walk(des):
    for file_name in files:
        local_path = os.path.join(root, file_name)
        rel_path = os.path.relpath(local_path, des)
        remote_path = os.path.join(des, rel_path).replace("\\", "/")

        if file_exists(remote_path):
            print(f"⏩ Bỏ qua (đã tồn tại): {remote_path}")
            continue

        # Tạo thư mục cha trên Samba nếu chưa có
        remote_folder = os.path.dirname(remote_path)
        parts = remote_folder.split("/")
        current_path = ""
        for part in parts:
            if part:
                current_path += f"/{part}"
                try:
                    conn.listPath(share_name, current_path)
                except:
                    conn.createDirectory(share_name, current_path)

        # Copy file
        print(f"📤 Đang upload: {remote_path}")
        with open(local_path, "rb") as f:
            conn.storeFile(share_name, remote_path, f)

conn.close()
print("✅ Hoàn tất sao chép các file chưa tồn tại.")


KeyboardInterrupt: 